In [1]:
URDF_PATH = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\URDF Model\a1.urdf"
SAVE_PATH = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\Project Arbeit Model"
LOGS_PATH = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\Project Arbeit Logs"
TRAINING_MODEL = r"D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\Project Arbeit Model\Training Model"

In [2]:
import os
import gymnasium as gym
import numpy as np
import pybullet as p
import pybullet_data
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from noise import pnoise2

In [3]:
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

if not os.path.exists(TRAINING_MODEL):
    os.makedirs(TRAINING_MODEL)
        
name = "baseline_1.zip"
name_ = "baseline_1" 
full_model_path = os.path.join(SAVE_PATH, name)
MODEL_NAME = name

In [4]:
# Constants to define training and visualisation.
GUI_MODE = False      # Set "True" to display pybullet in a window
EPISODE_LENGTH = 1000      # Number of steps for one training episode
MAXIMUM_LENGTH = 20_000_000     # Number of total steps for entire training
SIZE_OBSERVATION = 42
BOUND_ANGLE_MIN = np.array([-0.802851455917, -1.0471975512, -2.69653369433, -0.802851455917, -1.0471975512, -2.69653369433, -0.802851455917, -1.0471975512, -2.69653369433, -0.802851455917, -1.0471975512, -2.69653369433]) # Joint minimum angle (rad)
BOUND_ANGLE_MAX = np.array([0.802851455917, 4.18879020479, -0.916297857297, 0.802851455917, 4.18879020479, -0.916297857297, 0.802851455917, 4.18879020479, -0.916297857297, 0.802851455917, 4.18879020479, -0.916297857297]) # Joint maximum angle (rad)


In [5]:
class OpenCatGymEnv(gym.Env): ## We have created a custom class which inherits from the Env class of gym libraray
    """ Gymnasium environment (stable baselines 3) for OpenCat robots.
    """

    metadata = {'render.modes': ['human']} ## Just to show what kind of visualization we want. (there are 2-3 other modes also like rgb_array)

    def __init__(self, target_speed = 0.375): ## Constructor to setup everything
        self.episode_length = 1000
        self.control_frequency = 100  # Hz
        self.target_speed = target_speed
        self.step_counter = 0  # Counts steps in the current episode
        self.step_counter_session = 0 # May be used to track steps across all episodes (total training time).
        self.alpha1 = 0.04
        self.alpha2 = 20
        self.alive_bonus = 20 * target_speed
        self.vx_smooth = 0
        self.vy_smooth = 0
        self.wyaw_smooth = 0
        self.alpha_smooth = 0.2
        self.prev_action = np.zeros(12)
        self.max_vel = 25
        self.prev_torque = np.zeros(12)
        # In __init__:
        self.contact_avg = np.array([0.5, 0.5, 0.5, 0.5])

        
        
        self.default_pose = np.array([
                    -0.05, 0.8, -1.6,
                    0.05, 0.8, -1.6,
                    -0.05, 0.8, -1.6,
                    0.05, 0.8, -1.6 ])
        
                # With per-joint limits matching URDF:
        self.torque_limits = np.array([
            20.0, 55.0, 55.0,   # FR hip, upper, lower
            20.0, 55.0, 55.0,   # FL
            20.0, 55.0, 55.0,   # RR
            20.0, 55.0, 55.0,   # RL
        ])

        if GUI_MODE:
            p.connect(p.GUI)  # Opens visual window.
        else:
            # Use for training without visualisation (significantly faster).
            p.connect(p.DIRECT)  # Headless mode (no visuals) → faster for training.

    
        p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)
        p.resetDebugVisualizerCamera(cameraDistance=1.5,
                                cameraYaw=45,
                                cameraPitch=-30,
                                cameraTargetPosition=[0.3,0,0.1])

        # The action space are the 12 joint angles.
        self.action_space = gym.spaces.Box(low=-1,high=1,shape=(12,),dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=-1, high=1, shape=(SIZE_OBSERVATION,),dtype=np.float32)


    def reward(self, vx, vy, wyaw, torque, joint_vel,action):
        progress = min(self.step_counter_session / MAXIMUM_LENGTH, 1.0)
        # --- 1. Forward velocity (clipped target) ---
        r_forward = min(vx, self.target_speed)

        # --- 2. Lateral + yaw penalty ---
        r_stability = - (vy**2 + wyaw**2)

        # --- 3. Work (energy) ---
        r_work = -np.abs(np.dot(torque, joint_vel))

        # --- 4. Smoothness (torque change) ---
        r_smooth_tau = -np.sum((torque - self.prev_torque)**2)

        # --- 5. Action smoothness ---
        r_smooth_action = -np.sum((action - self.prev_action)**2)

        # --- 6. Action magnitude ---
        r_action_mag = -np.sum(action**2)

        # --- 7. Joint velocity penalty ---
        r_joint_vel = -np.sum(joint_vel**2)

        # --- 8. Orientation penalty ---
        pos, quat = p.getBasePositionAndOrientation(self.robot_id)
        roll, pitch, _ = p.getEulerFromQuaternion(quat)
        r_orientation = -(roll**2 + pitch**2)

        # --- 9. Vertical velocity penalty ---
        base_vel = p.getBaseVelocity(self.robot_id)[0]
        vz = base_vel[2]
        r_z_vel = -(vz**2)

        # --- 10. Foot contact / slip (simplified) ---
        # If foot in contact → penalize velocity
        foot_vel_penalty = 0
        paw_links = [5, 10, 15, 20]

        # contacts  = []
        # for link in paw_links:
        #     c = p.getContactPoints(self.robot_id, self.plane_id, link)
        #     contacts.append(1.0 if len(c) > 0 else 0.0)
        # contacts = np.array(contacts)

        # # Running average of contacts (add self.contact_avg to __init__)
        # self.contact_avg = 0.95 * self.contact_avg + 0.05 * contacts

        # Penalize deviation from 0.5 contact rate
        # r_contact_balance = -np.sum((self.contact_avg - 0.5) ** 2)

        #         # Add to reward function:
        # thigh_joints = np.array([joint_vel[1], joint_vel[4],
        #                         joint_vel[7], joint_vel[10]])
        # r_thigh_activity = np.sum(np.abs(thigh_joints))   # reward thigh movement
        contacts = []

        for link in paw_links:
            contact = p.getContactPoints(self.robot_id, self.plane_id, link)
            if len(contact)>0:
                contacts.append(1)
            else:
                contacts.append(0)

            if len(contact) > 0:  # foot in contact
                link_state = p.getLinkState(self.robot_id, link, computeLinkVelocity=1)

                foot_vel = np.array(link_state[6])  # [vx, vy, vz]

                # 🔥 ONLY tangential velocity
                foot_vel_t = foot_vel[:2]   # x, y

                # 🔥 L2 norm (NOT squared)
                foot_vel_penalty += np.linalg.norm(foot_vel_t)

        r_foot_slip = -foot_vel_penalty
        num_contacts = np.sum(contacts)

        r_contact = -max(0, abs(num_contacts - 2.5) - 0.5)**2

        # ADD: penalize asymmetric leg work
        # Front legs: joints 0-5, Rear legs: joints 6-11
        # front_work = np.sum(np.abs(torque[:6] * joint_vel[:6]))
        # rear_work  = np.sum(np.abs(torque[6:] * joint_vel[6:]))
        # r_symmetry = -float(abs(front_work - rear_work))


        work_legs = []
        for i in range(4):
            start = i * 3
            end = start + 3
            work = np.sum(np.abs(torque[start:end] * joint_vel[start:end]))
            work_legs.append(work)

        work_legs = np.array(work_legs)

        r_leg_balance = -np.std(work_legs)



        reward = (
            20.0 * r_forward +
            progress*(2.0  * r_stability +
            0.002 * r_work +
            # 5.0*r_contact +
            0.001 * r_smooth_tau +
            0.001 * r_smooth_action +
            0.07  * r_action_mag +
            0.002 * r_joint_vel +
            1.5   * r_orientation +
            2.0   * r_z_vel +
            0.8   * r_foot_slip +
            2.0   * r_leg_balance)
        )

        return reward
        
   
    def step(self, action):
        self.step_counter +=1
        if GUI_MODE:
            p.configureDebugVisualizer(p.COV_ENABLE_SINGLE_STEP_RENDERING) ## Render only after 1 step
        # Current joint states
        joint_states = p.getJointStates(self.robot_id, self.joint_id)
        joint_angles = np.array([s[0] for s in joint_states])
        joint_vel = np.array([s[1] for s in joint_states])

        delta_limits = np.array([
            0.10, 0.25, 0.20,   # FR — all reduced to stay within torque limits
            0.10, 0.25, 0.20,   # FL
            0.10, 0.25, 0.20,   # RR
            0.10, 0.25, 0.20,   # RL
        ])
        action = action*delta_limits
        target_angles = self.default_pose + action

        Kp = 200
        Kd = 1.0
        torque = self.motor_strength*(Kp * (target_angles - joint_angles) - Kd * joint_vel)

        torque = np.clip(torque, -self.torque_limits, self.torque_limits)
        p.setJointMotorControlArray(
            self.robot_id,
            self.joint_id,
            controlMode=p.TORQUE_CONTROL,
            forces=torque.tolist()
        )

        for _ in range(2):
            p.stepSimulation()

        base_vel = p.getBaseVelocity(self.robot_id)[0]
        vx = base_vel[0]
        vy = base_vel[1]
    
        _, ang_vel = p.getBaseVelocity(self.robot_id)
        wx, wy, wyaw = ang_vel  # roll rate, pitch rate, yaw rate
        self.vx_smooth = self.alpha_smooth * vx + (1 - self.alpha_smooth) * self.vx_smooth
        self.vy_smooth = self.alpha_smooth * vy + (1 - self.alpha_smooth) * self.vy_smooth
        self.wyaw_smooth = self.alpha_smooth * wyaw + (1 - self.alpha_smooth) * self.wyaw_smooth
        reward = self.reward(self.vx_smooth, self.vy_smooth, self.wyaw_smooth, torque, joint_vel,action)

        # Set state of the current state.
        terminated = False
        truncated = False

        if self.step_counter > self.episode_length:
            self.step_counter_session += self.step_counter
            terminated = False
            truncated = True

        elif self.is_fallen(): # Robot fell
            self.step_counter_session += self.step_counter
            terminated = True
            truncated = False

        observation = self._get_observation()
        self.prev_action = action.copy()
        self.prev_torque = torque.copy()

        return observation.astype(np.float32), reward, terminated, truncated, {}
    
    def generate_fractal_terrain(self, size=256,
                            scale=1.0,
                            octaves=2,
                            lacunarity=2.0,
                            gain=0.25,
                            frequency=10,
                            amplitude=0.23):

            heightfield = np.zeros((size, size))

            for i in range(size):
                for j in range(size):

                    x = i / size * frequency
                    y = j / size * frequency

                    height = pnoise2(
                        x,
                        y,
                        octaves=octaves,
                        persistence=gain,
                        lacunarity=lacunarity,
                        repeatx=1024,
                        repeaty=1024,
                        base=0
                    )

                    heightfield[i][j] = height

            heightfield = heightfield * amplitude

            return heightfield
    

    def create_heightfield(self, heightfield):

        size = heightfield.shape[0]

        terrainShape = p.createCollisionShape(
            shapeType=p.GEOM_HEIGHTFIELD,
            meshScale=[0.05, 0.05, 1],
            heightfieldTextureScaling=(size - 1) / 2,
            heightfieldData=heightfield.flatten(),
            numHeightfieldRows=size,
            numHeightfieldColumns=size
        )

        terrain = p.createMultiBody(0, terrainShape)

        p.resetBasePositionAndOrientation(terrain, [0, 0, 0], [0,0,0,1])

        return terrain

    def reset(self, seed = None, options=None):
        progress = min(self.step_counter_session / MAXIMUM_LENGTH, 1.0)
        super().reset(seed=seed)
        # 3. In reset() — add prev_torque reset and increase settle to 300 steps
        self.prev_torque = np.zeros(12)
        # In reset():
        self.contact_avg = np.array([0.5, 0.5, 0.5, 0.5])

        self.prev_action = np.zeros(12)
        self.vx_smooth = 0
        self.vy_smooth = 0
        self.wyaw_smooth = 0
        self.step_counter = 0
        p.resetSimulation()
        # Disable rendering during loading.
        p.configureDebugVisualizer(p.COV_ENABLE_RENDERING,0) 
        p.setGravity(0,0,-9.81)
        p.setPhysicsEngineParameter(numSolverIterations=50, numSubSteps=1)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setTimeStep(1.0 / 240.0)
        #plane
        # heightfield = self.generate_fractal_terrain(size=256, scale=1, octaves=2, lacunarity=2.0, gain=0.25, frequency=10, amplitude=0.23)
        # plane_id = self.create_heightfield(heightfield)
        self.plane_id = p.loadURDF("plane.urdf")
        # p.changeDynamics(self.plane_id, -1, lateralFriction=5)

        start_pos = [0,0,0.4]
        start_orient = p.getQuaternionFromEuler([0,0,0])

        urdf_path = URDF_PATH
        self.robot_id = p.loadURDF(urdf_path,
                                   start_pos, start_orient,
                                   flags=p.URDF_USE_SELF_COLLISION) ## This will enable the collision checking between robot links
        

        # if progress > 0.0625:  # First 50% of training → no randomization
        #     friction = np.random.uniform(0.6, 1.2)
        #     payload = np.random.uniform(0.0, 0.5)
        #     mass = p.getDynamicsInfo(self.robot_id, -1)[0]

        #     p.changeDynamics(self.robot_id,-1,mass=mass + payload)
        #     p.changeDynamics(self.plane_id, -1, lateralFriction=friction)
        #     self.motor_strength = np.random.uniform(0.95, 1.05)
        # else:
        self.motor_strength = 1

        # Initialize urdf links and joints.
        self.joint_id = []
        for j in range(p.getNumJoints(self.robot_id)):
            info = p.getJointInfo(self.robot_id, j)
            joint_name = info[1]
            joint_type = info[2]

            if (joint_type == p.JOINT_PRISMATIC
                or joint_type == p.JOINT_REVOLUTE):
                self.joint_id.append(j)


        for j in self.joint_id:
            p.changeDynamics(
                self.robot_id,
                j,
                linearDamping=0.04,

                angularDamping=0.04)
            
         # ADD this before resetJointState
        p.resetBasePositionAndOrientation(
            self.robot_id, [0, 0, 0.40],
            p.getQuaternionFromEuler([0, 0, 0])
        )
        p.resetBaseVelocity(self.robot_id,
            linearVelocity=[0, 0, 0],
            angularVelocity=[0, 0, 0]
        )   

#         p.resetBaseVelocity(self.robot_id,
#     linearVelocity=[np.random.uniform(0.1, 0.25), 0, 0],
#     angularVelocity=[0, 0, 0]
# )
                    

        for j, q in zip(self.joint_id, self.default_pose):
            p.resetJointState(self.robot_id, j, targetValue=q, targetVelocity=0.0)


        # Settle — 100 steps confirmed sufficient at 400 Hz
        for _ in range(100):
            p.setJointMotorControlArray(
                self.robot_id, self.joint_id,
                controlMode=p.POSITION_CONTROL,
                targetPositions=self.default_pose,
                forces        =[200.0] * len(self.joint_id),
                positionGains =[0.5]   * len(self.joint_id),
                velocityGains =[1.0]   * len(self.joint_id),
            )
            p.stepSimulation()

        # Disable motors — hand off to torque control
        p.setJointMotorControlArray(
            self.robot_id, self.joint_id,
            controlMode=p.VELOCITY_CONTROL,
            forces=[0.0] * len(self.joint_id)
        )


        observation = self._get_observation()    

        p.configureDebugVisualizer(p.COV_ENABLE_RENDERING,1)
        return observation.astype(np.float32), {}


    def render(self, mode='human'):
        if GUI_MODE:
            p.configureDebugVisualizer(p.COV_ENABLE_SINGLE_STEP_RENDERING) ## Render only after 1 step



    def close(self):
        p.disconnect()


    def is_fallen(self):
        pos, orient = p.getBasePositionAndOrientation(self.robot_id)
        orient = p.getEulerFromQuaternion(orient)
        is_fallen = (np.fabs(orient[0]) > 0.4
                    or np.fabs(orient[1]) > 0.2
                    or pos[2]<0.1)

        return is_fallen
    
    def _get_observation(self):
        joint_states = p.getJointStates(self.robot_id, self.joint_id)
        # Joint angles
        joint_angles = np.array([s[0] for s in joint_states])
        joint_low = np.array(BOUND_ANGLE_MIN)
        joint_high = np.array(BOUND_ANGLE_MAX)

        joint_angles_norm = 2 * (joint_angles - joint_low) / (joint_high - joint_low) - 1
        joint_angles_norm = np.clip(joint_angles_norm, -1.0, 1.0)
        # Joint velocities
        joint_vel = np.array([s[1] for s in joint_states])
        joint_vel_norm = np.clip(joint_vel / self.max_vel, -1, 1)
        # Base orientation
        pos, quat = p.getBasePositionAndOrientation(self.robot_id)
        roll, pitch, _ = p.getEulerFromQuaternion(quat)
        torso = np.array([roll, pitch])
        torso_norm = torso / np.pi
        # Foot contacts
        paw_links = [5,10,15,20]
        contacts = []
        for idx in paw_links:
            contact = p.getContactPoints(bodyA=self.robot_id,bodyB= self.plane_id, linkIndexA=idx)
            contacts.append(1 if len(contact) > 0 else 0)


        contacts = np.array(contacts)
        observation = np.concatenate(
            (joint_angles_norm,joint_vel_norm,torso_norm,contacts, self.prev_action))

        return observation


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize
from stable_baselines3.common.logger import configure
from stable_baselines3.common.callbacks import CheckpointCallback

In [ ]:
##Code to train the model
STATS_NAME = f"stats_{name_}.pkl"
full_stats_path = os.path.join(SAVE_PATH, STATS_NAME)

if __name__ == "__main__":
    # Set up number of parallel environments
    # def linear_schedule(initial_lr):
    #     def func(progress_remaining):
    #         return progress_remaining * initial_lr
    #     return func
    log_path = f"{LOGS_PATH}/{MODEL_NAME}/"
    new_logger = configure(log_path, ["stdout", "csv", "tensorboard"])
    parallel_envs = 8
    envs = make_vec_env(OpenCatGymEnv, n_envs=parallel_envs, vec_env_cls=SubprocVecEnv) # only for PPO
    envs = VecNormalize(envs, norm_obs=True, norm_reward=True, clip_obs=10.0,       # prevents observation explosion during early training
        clip_reward=10.0,
        gamma=0.99)

    # -----------------------------
    # NEW: Setup Checkpoint Callback
    # -----------------------------
    # save_freq is per environment step. 
    # To save exactly at 5,000,000 total steps with 8 envs:
    SAVE_FREQ = 1_000_000 // parallel_envs

    checkpoint_callback = CheckpointCallback(
        save_freq=SAVE_FREQ,
        save_path=TRAINING_MODEL,
        name_prefix=name_,        # This keeps your "ninth" name
        save_vecnormalize=True,   # Saves the .pkl stats automatically
        verbose=2
    )

    # Change architecture of neural network to two hidden layers of size 256
    #custom_arch = dict(net_arch=[256,256])   # ≈ std 0.22

    # Define PPO agent and train
    model = PPO('MlpPolicy', envs,
                seed=42,
                learning_rate= 1e-4,
                # rollout size
                n_steps=2048,

                # PPO training settings
                batch_size=256,
                n_epochs=10,

                # PPO clipping
                clip_range=0.3,
                # Entropy — encourages exploration, important for locomotion
                # ent_coef=entropy_schedule,
                # max_grad_norm=0.5,
                # GAE
                gamma=0.99,
                gae_lambda=0.95,
                # Network + std initialization
                policy_kwargs=dict(
                    # log_std_init=-1.0,      # std starts at 0.36 instead of 1.0
                    net_arch=[256, 256]     # larger network for complex locomotion
                ),
                verbose=1)
    model.set_logger(new_logger)

    model.learn(total_timesteps=MAXIMUM_LENGTH, callback=checkpoint_callback)

    model.save(full_model_path)
    envs.save(full_stats_path)

Logging to D:\Project Arbeit\Steps-Generation-Using-Reinforcement-Learning\Project Arbeit Logs/baseline_1.zip/
Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 575      |
|    ep_rew_mean     | -574     |
| time/              |          |
|    fps             | 2883     |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 32768    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 538          |
|    ep_rew_mean          | -708         |
| time/                   |              |
|    fps                  | 2523         |
|    iterations           | 2            |
|    time_elapsed         | 25           |
|    total_timesteps      | 65536        |
| train/                  |              |
|    approx_kl            | 0.0046691825 |
|    clip_fraction        | 0.079        |
|    clip_range 

<!-- Here, I will give you the full arcitecture of PPO. -->
